# Task 3.1.6: Object Tracking using Dense Optical Flow

In [1]:
# Importing required libraries, which include:
# OpenCV library used for the many tasks in computer vision, from loading images, to processing, detecting, shapes, tracking objects etc.:
import cv2
import os
import numpy as np

In [2]:
# TASK1: Dense optical flow. 

# Hint: You can follow next steps: 
#       Calculate dense optical flow between consecutive frames
#       Encode the flow information using Hue and Value channels.
#       Detect moving objects by finding contours. 
#       Draw a bounding box around each contour.
#       Display the frame.

# Set the path to the directory containing the frames
frame_dir = "./data/got10k/sequence_example2/"
frame_filenames = sorted(os.listdir(frame_dir))

prev_frame = cv2.imread(os.path.join(frame_dir, frame_filenames[0]))
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)


# Perform tracking using dense optical flow

for filename in frame_filenames[1:]:
    frame = cv2.imread(os.path.join(frame_dir, filename))
    
    if frame is None:
        continue
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    
    flow = cv2.calcOpticalFlowFarneback(
        prev_gray,
        gray,
        None,
        0.5,
        3,
        15,
        3,
        5,
        1.2,
        0
    )
    
    
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    hsv = np.zeros_like(frame)
    hsv[..., 0] = angle * 180 / np.pi / 2
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)

    flow_visual = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    
        # detect moving objects by thresholding motion magnitude
    motion_mask = cv2.threshold(
        hsv[..., 2],
        25,
        255,
        cv2.THRESH_BINARY
    )[1]

    # find contours
    contours, _ = cv2.findContours(
        motion_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # draw a bounding box around each contour
    for contour in contours:
        if cv2.contourArea(contour) > 100:
            x, y, w, h = cv2.boundingRect(contour)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # display the frame
    cv2.imshow("Dense optical flow tracking", frame)
    cv2.imshow("Dense optical flow visualization", flow_visual)

    key = cv2.waitKey(30) & 0xFF

    if key == ord("q") or cv2.getWindowProperty("Dense optical flow tracking", cv2.WND_PROP_VISIBLE) < 1:
        break

    prev_gray = gray.copy()

cv2.destroyAllWindows()

for _ in range(20):
    cv2.waitKey(1)
    
    
    

: 